In [1]:
!pip install gradio pandas


In [2]:
import pandas as pd
import os
from datetime import datetime
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_PATH = "/content/drive/MyDrive/Cloud_Attendance"

os.makedirs(PROJECT_PATH, exist_ok=True)

print("Cloud storage connected!")
print("Project folder:", PROJECT_PATH)

Cloud storage connected!
Project folder: /content/drive/MyDrive/Cloud_Attendance


In [5]:
student_file = os.path.join(PROJECT_PATH, "students.csv")

if not os.path.exists(student_file):
    students = pd.DataFrame(columns=[
        "Student_ID",
        "Name",
        "Department",
        "Year"
    ])

    students.to_csv(student_file, index=False)

print("Student database ready!")

Student database ready!


In [6]:
students = pd.DataFrame([
    ["CSE001", "Arun", "CSE", 3],
    ["CSE002", "Priya", "CSE", 3],
    ["CSE003", "Rahul", "CSE", 3],
    ["CSE004", "Anjali", "CSE", 3],
    ["CSE005", "Karthik", "CSE", 3]
], columns=[
    "Student_ID",
    "Name",
    "Department",
    "Year"
])

students.to_csv(student_file, index=False)

print("Students added successfully!")
display(students)

Students added successfully!


,Student_ID,Name,Department,Year
0,CSE001,Arun,CSE,3
1,CSE002,Priya,CSE,3
2,CSE003,Rahul,CSE,3
3,CSE004,Anjali,CSE,3
4,CSE005,Karthik,CSE,3


In [7]:
attendance_file = os.path.join(PROJECT_PATH, "attendance.csv")

if not os.path.exists(attendance_file):
    attendance = pd.DataFrame(columns=[
        "Date",
        "Student_ID",
        "Name",
        "Status"
    ])

    attendance.to_csv(attendance_file, index=False)

print("Attendance database ready!")

Attendance database ready!


In [8]:
def mark_attendance(student_id, status):

    students = pd.read_csv(student_file)

    student = students[
        students["Student_ID"] == student_id
    ]

    if student.empty:
        return "Student ID not found!"

    name = student.iloc[0]["Name"]

    date = datetime.now().strftime("%Y-%m-%d")

    attendance = pd.read_csv(attendance_file)

    # Prevent duplicate attendance
    existing = attendance[
        (attendance["Date"] == date) &
        (attendance["Student_ID"] == student_id)
    ]

    if not existing.empty:
        return "Attendance already marked for today!"

    new_record = pd.DataFrame([{
        "Date": date,
        "Student_ID": student_id,
        "Name": name,
        "Status": status
    }])

    attendance = pd.concat(
        [attendance, new_record],
        ignore_index=True
    )

    attendance.to_csv(
        attendance_file,
        index=False
    )

    return f"Attendance marked: {name} - {status}"

In [9]:
print(mark_attendance("CSE001", "Present"))

Attendance marked: Arun - Present


In [10]:
print(mark_attendance("CSE002", "Absent"))

Attendance marked: Priya - Absent


In [11]:
pd.read_csv(attendance_file)

,Date,Student_ID,Name,Status
0,2026-10-04,CSE001,Arun,Present
1,2026-10-04,CSE002,Priya,Absent


In [12]:
def attendance_percentage(student_id):

    attendance = pd.read_csv(attendance_file)

    student_attendance = attendance[
        attendance["Student_ID"] == student_id
    ]

    if student_attendance.empty:
        return "No attendance records found."

    total = len(student_attendance)

    present = len(
        student_attendance[
            student_attendance["Status"] == "Present"
        ]
    )

    percentage = (present / total) * 100

    return f"Attendance Percentage: {percentage:.2f}%"

In [13]:
print(attendance_percentage("CSE001"))

Attendance Percentage: 100.00%


In [14]:
def attendance_report():

    attendance = pd.read_csv(attendance_file)

    if attendance.empty:
        return "No attendance records available."

    report = attendance.groupby(
        ["Student_ID", "Name"]
    ).agg(
        Total_Days=("Status", "count"),
        Present=("Status", lambda x: (x == "Present").sum()),
        Absent=("Status", lambda x: (x == "Absent").sum())
    ).reset_index()

    report["Attendance_Percentage"] = (
        report["Present"] /
        report["Total_Days"] * 100
    ).round(2)

    return report

In [15]:
display(attendance_report())

,Student_ID,Name,Total_Days,Present,Absent,Attendance_Percentage
0,CSE001,Arun,1,1,0,100.0
1,CSE002,Priya,1,0,1,0.0


In [16]:
!pip install -q gradio

In [17]:
import gradio as gr

In [18]:
with gr.Blocks(title="Cloud Attendance System") as app:

    gr.Markdown(
        "# ☁️ Cloud-Based Attendance System"
    )

    gr.Markdown(
        "Student Attendance Management System"
    )

    with gr.Tab("Mark Attendance"):

        student_id = gr.Textbox(
            label="Student ID",
            placeholder="Example: CSE001"
        )

        status = gr.Radio(
            ["Present", "Absent"],
            label="Attendance Status"
        )

        mark_button = gr.Button(
            "Mark Attendance"
        )

        result = gr.Textbox(
            label="Result"
        )

        mark_button.click(
            mark_attendance,
            inputs=[student_id, status],
            outputs=result
        )

    with gr.Tab("Attendance Percentage"):

        percentage_id = gr.Textbox(
            label="Student ID"
        )

        percentage_button = gr.Button(
            "Check Percentage"
        )

        percentage_result = gr.Textbox()

        percentage_button.click(
            attendance_percentage,
            inputs=percentage_id,
            outputs=percentage_result
        )

    with gr.Tab("Attendance Report"):

        report_button = gr.Button(
            "Generate Report"
        )

        report_output = gr.Dataframe()

        report_button.click(
            attendance_report,
            outputs=report_output
        )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b207e298d6216ca37.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
